In [ ]:
## Part 1: Collect

In [7]:
import json
import random
import time
from datetime import date, datetime, timedelta
from pathlib import Path
from urllib.parse import quote

import requests

DATA_DIR = Path("data")
RAW = DATA_DIR / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# Wikimedia asks for a User-Agent that identifies you and gives contact info
HEADERS = {"User-Agent": "CU Boulder student project (Daphnne Chacon; dach2819@colorado.edu)"}
WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKIDATA_API = "https://www.wikidata.org/w/api.php"
PV_BASE = ("https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
           "en.wikipedia/all-access/user")

PAUSE = 1           # seconds to wait after every request
SAMPLE_SIZE = 300   # people sampled from Category:2024 deaths
SEED = 4617         # so the random sample is reproducible

session = requests.Session()
session.headers.update(HEADERS)

def get_json(url, params=None):
    """Polite GET: identifying User-Agent, a pause after every request, stop on errors."""
    r = session.get(url, params=params, timeout=30)
    time.sleep(PAUSE)
    r.raise_for_status()
    return r.json()

def save(obj, name):
    (RAW / name).write_text(json.dumps(obj, indent=1))

In [8]:
params = {
    "action": "query", "list": "categorymembers",
    "cmtitle": "Category:2024 deaths", "cmnamespace": 0, "cmlimit": "max",
    "format": "json", "formatversion": 2,
}
members = []
while True:
    data = get_json(WIKI_API, params)
    members.extend(data["query"]["categorymembers"])
    if "continue" not in data:
        break
    params.update(data["continue"])

save(members, "category_members.json")
print("Articles in Category:2024 deaths:", len(members))

Articles in Category:2024 deaths: 11939


In [9]:
death_claims = {}
qid_list = list(qids.values())
for i in range(0, len(qid_list), 50):
    data = get_json(WIKIDATA_API, {
        "action": "wbgetentities", "ids": "|".join(qid_list[i:i + 50]),
        "props": "claims", "format": "json",
    })
    for qid, entity in data["entities"].items():
        death_claims[qid] = entity.get("claims", {}).get("P570", [])   # P570 = date of death

save(death_claims, "death_dates.json")

def parse_death_date(claims):
    """Return the death date only if Wikidata records it to the exact day."""
    for claim in claims:
        value = claim["mainsnak"].get("datavalue", {}).get("value")
        if value and value.get("precision") == 11:   # 11 means day-level precision
            return datetime.strptime(value["time"][1:11], "%Y-%m-%d").date()
    return None

print("People with an exact death date:",
      sum(parse_death_date(c) is not None for c in death_claims.values()))

People with an exact death date: 275


In [10]:
pv_end = date.today() - timedelta(days=2)   # pageviews take a day or two to appear

def get_pageviews(title, start, end):
    article = quote(title.replace(" ", "_"), safe="")
    r = session.get(f"{PV_BASE}/{article}/daily/{start:%Y%m%d}/{end:%Y%m%d}", timeout=30)
    time.sleep(PAUSE)
    if r.status_code == 404:   # no recorded views in this window
        return []
    r.raise_for_status()
    return r.json()["items"]

revisions, pageviews = {}, {}
for n, (title, qid) in enumerate(qids.items(), start=1):
    dod = parse_death_date(death_claims.get(qid, []))
    if dod is None or dod.year != 2024:
        continue
    # the last revision before the day they died, so length isn't inflated by obituary edits
    revisions[title] = get_json(WIKI_API, {
        "action": "query", "prop": "revisions", "titles": title,
        "rvprop": "size|timestamp", "rvlimit": 1,
        "rvstart": f"{dod}T00:00:00Z", "rvdir": "older",
        "format": "json", "formatversion": 2,
    })
    pageviews[title] = get_pageviews(title, dod - timedelta(days=90), pv_end)
    if n % 25 == 0:
        print(f"{n} of {len(qids)} done")

save(revisions, "revisions.json")
save(pageviews, "pageviews.json")
save({"collected_on": str(date.today()), "pageviews_end": str(pv_end),
      "category": "Category:2024 deaths", "sample_size": SAMPLE_SIZE, "seed": SEED},
     "collection_info.json")
print("People with pageviews collected:", len(pageviews))

25 of 292 done
50 of 292 done
75 of 292 done
100 of 292 done
125 of 292 done
150 of 292 done
175 of 292 done
200 of 292 done
225 of 292 done
250 of 292 done
275 of 292 done
People with pageviews collected: 274


In [11]:
import pandas as pd

RAW, CLEAN, SAMPLE = DATA_DIR / "raw", DATA_DIR / "clean", DATA_DIR / "sample"
CLEAN.mkdir(exist_ok=True)
SAMPLE.mkdir(exist_ok=True)
MIN_PEAK = 500   # drop people whose death barely registered

def load(name):
    return json.loads((RAW / name).read_text())

info = load("collection_info.json")
qids = load("wikidata_ids.json")
death_claims = load("death_dates.json")
revisions = load("revisions.json")
pageviews = load("pageviews.json")
end_date = pd.to_datetime(info["pageviews_end"])

def parse_death_date(claims):
    for claim in claims:
        value = claim["mainsnak"].get("datavalue", {}).get("value")
        if value and value.get("precision") == 11:
            return pd.to_datetime(value["time"][1:11])
    return pd.NaT

print(info)

{'collected_on': '2026-11-09', 'pageviews_end': '2026-11-07', 'category': 'Category:2024 deaths', 'sample_size': 300, 'seed': 4617}


In [12]:
rows = []
for title, qid in qids.items():
    pages = revisions.get(title, {}).get("query", {}).get("pages", [{}])
    revs = pages[0].get("revisions", [])
    rows.append({
        "title": title,
        "wikidata_id": qid,
        "death_date": parse_death_date(death_claims.get(qid, [])),
        "length_before_death": revs[0]["size"] if revs else None,
    })
people = pd.DataFrame(rows)
print("Sampled people with a Wikidata ID:", len(people))

people = people.dropna(subset=["death_date"])
people = people[people["death_date"].dt.year == 2024]
print("With an exact 2024 death date:", len(people))

people = people.dropna(subset=["length_before_death"])
people["length_before_death"] = people["length_before_death"].astype(int)
print("With an article that existed before they died:", len(people))

Sampled people with a Wikidata ID: 292
With an exact 2024 death date: 274
With an article that existed before they died: 209


In [13]:
frames = []
for row in people.itertuples():
    observed = pd.Series(
        {pd.to_datetime(i["timestamp"][:8], format="%Y%m%d"): i["views"]
         for i in pageviews.get(row.title, [])},
        dtype="int64",
    )
    all_days = pd.date_range(row.death_date - pd.Timedelta(days=90), end_date, freq="D")
    frames.append(pd.DataFrame({
        "title": row.title,
        "date": all_days,
        "days_since_death": (all_days - row.death_date).days,
        "views": observed.reindex(all_days, fill_value=0).values,
    }))
daily = pd.concat(frames, ignore_index=True)
print("Daily rows:", len(daily))

Daily rows: 197540


In [14]:
before = daily[daily["days_since_death"].between(-90, -1)]
after = daily[daily["days_since_death"].between(0, 30)]

people = people.set_index("title")
people["baseline_views"] = before.groupby("title")["views"].median()
people["peak_views"] = after.groupby("title")["views"].max()
people["peak_date"] = after.loc[after.groupby("title")["views"].idxmax()].set_index("title")["date"]
people = people.reset_index()

people = people[people["peak_views"] >= MIN_PEAK]
daily = daily[daily["title"].isin(people["title"])]
print(f"With a peak of at least {MIN_PEAK} views:", len(people))

With a peak of at least 500 views: 133


In [15]:
people["length_group"] = pd.qcut(people["length_before_death"], 3,
                                 labels=["short", "medium", "long"])
print(people.groupby("length_group", observed=True)["length_before_death"].agg(["count", "min", "max"]))

              count   min    max
length_group                    
short            45    38   4020
medium           43  4126   8030
long             45  8263  86240


In [16]:
people.to_csv(CLEAN / "people.csv", index=False)
daily.to_csv(CLEAN / "pageviews_daily.csv", index=False)

sample_titles = people.sample(10, random_state=4617)["title"]
people[people["title"].isin(sample_titles)].to_csv(SAMPLE / "people_sample.csv", index=False)
daily[daily["title"].isin(sample_titles) & daily["days_since_death"].between(-14, 60)] \
    .to_csv(SAMPLE / "pageviews_daily_sample.csv", index=False)

print("people.csv:", people.shape, "| pageviews_daily.csv:", daily.shape)
display(people.head())
display(daily.head())

people.csv: (133, 8) | pageviews_daily.csv: (124520, 4)


,title,wikidata_id,death_date,length_before_death,baseline_views,peak_views,peak_date,length_group
0,Alice Munro,Q234819,2024-05-13,42714,485.5,75817,2024-05-14,long
1,Anita Bryant,Q240623,2024-12-16,52863,751.0,336039,2025-01-10,long
3,Barbara Dane,Q4858851,2024-10-20,15437,59.0,6918,2024-10-22,long
4,Aleksandr Medved,Q435761,2024-09-02,9008,77.0,4930,2024-09-02,long
5,Bernard Marcus,Q2898293,2024-11-04,18644,330.0,65913,2024-11-05,long


,title,date,days_since_death,views
0,Alice Munro,2024-02-13,-90,582
1,Alice Munro,2024-02-14,-89,492
2,Alice Munro,2024-02-15,-88,475
3,Alice Munro,2024-02-16,-87,473
4,Alice Munro,2024-02-17,-86,453
